# Phase 1 experiments: vLLM v3 (0.33.0)
Dedicated vLLM notebook with opt-in inference parameters and measured-window
server/client telemetry, following the SGLang v3 reporting contract.

Uses versioned `vllm_policy_v3.py`, `runtime_vllm_v3.py` and `telemetry_vllm_v3.py`.
Existing notebooks and shared runners are unchanged. Run setup cells in order.
GPU execution happens remotely in Jupyter; CPU simulation checks wiring only.
The exact target is vLLM 0.33.0; other builds require source/adapter review.
The batch Python entry point does not inherit this notebook's v3 configuration.


## Differences from the v2 lab
See [the parity checklist](phase1_v2_parity.md). Independent optional cells below restore Pareto/constraint searches, additional pruners, calibration diagnostics, and pinned HF/speculative setup. The supervised default scalar queue remains unchanged. Untuned baselines and full v2 reporting remain separate. Phase A defaults to the same target as B; explicit HF setup changes only Phase B.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "llm_optimizer").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from llm_optimizer.experiment.phase1 import plot_group
from llm_optimizer.experiment.vllm_policy_v3 import (
    VLLMRunnerV3, DEFAULT_OPTIONAL_PARAMS, native_command,
    verify_runtime, runtime_help, audit_command,
)
from llm_optimizer.experiment.telemetry_vllm_v3 import DEFAULT_TELEMETRY
import copy


## VM, model, workload, and budget controls
Change `FRAMEWORKS` to a single framework on dedicated VMs, or all three on the comparison VM. All frameworks on one VM run sequentially and must be installed compatibly. Use a separate campaign name for independent replications. Changing any search setting creates a new configuration hash; restarts with identical settings resume existing studies. All campaigns on a VM must share `RUN_ROOT` and the same GPU assignment.
`SAMPLER_NAMES` selects GP (`gp`), TPE (`tpe`), and multivariate TPE (`tpe_multivariate`). GP requires the engine-compatible PyTorch installation; unavailable GP is explicitly reported and recorded as skipped, never replaced by TPE. Each combination of sampler, seed, framework, concurrency, objective and pruner has **20 attempted trials** by default. Failed/pruned attempts count toward that budget. Three samplers and three seeds therefore mean 180 attempts for a single concurrency/objective/framework/pruner, not 20 total. Optional `sampler_names=["gp"]` on a group call restricts that cell.

| Concurrency | Requests per trial | Default trial attempts per study |
|---|---|---|
| 1 | 32 | 20 |
| 8 | 62 | 20 |
| 16 | 125 | 20 |
| 32 | 250 | 20 |
| 128 | 1000 | 20 |

The loaded request/concurrency ratio is approximately 1000/128. Concurrency 1 deliberately uses 32 requests. `REQUESTS_BY_CONCURRENCY` is used by both A and B; explicit validation counts remain independently configured at 1000. Pruning observation floors scale from v2's 400/1000 ratio: 12, 24, 50, 100, 400 respectively. With small request counts, tail-latency estimates are noisy; the separate validation budget remains necessary.


In [ ]:
CAMPAIGN = "phase1-vllm-v3"
RUN_NAME = "fp8-clean-retry-01"  # W&B display-name prefix; keep campaign unchanged
SAVE_REQUEST_OUTPUTS = True  # persist generated_texts in local *-measured.jsonl files
FRAMEWORKS = ["vllm"]  # or ["vllm", "sglang", "trtllm"]
RUN_REAL_GPU = True
CONCURRENCIES = [1, 8, 16, 32, 128]
SEEDS = [11, 29, 47]
RUN_ROOT = ROOT / "experiment_results" / "phase1"
CAMPAIGN_HOURS = 168
GROUP_TIMEOUT_SECONDS = 6 * 3600  # includes all frameworks/seeds in a group
MAX_RESTARTS = 2  # bounded retries after a crashed/timed-out group
MODEL = "Qwen/Qwen3.5-4B"  # same model across A/B isolates search-space effects
MODEL_SNAPSHOT = "/workspace/models/pinned-model-snapshot"
TRIALS = 20  # attempts PER sampler/seed/framework/concurrency/pruner
SAMPLER_NAMES = ["tpe", "tpe_multivariate"]  # select any nonempty subset
PARETO_SAMPLERS = [*SAMPLER_NAMES, "nsga2"]  # available to optional Pareto calls
REQUESTS_BY_CONCURRENCY = {1: 32, 8: 62, 16: 125, 32: 250, 128: 1000}
# Preserve the v2 400/1000 pruning-observation fraction at each request budget.
PRUNING_MIN_COMPLETED_REQUESTS = {
    c: max(1, int(n * 0.4)) for c, n in REQUESTS_BY_CONCURRENCY.items()
}
SAMPLER_SETTINGS = {
    "startup_single": 4, "startup_multi": 10,
    "n_ei_candidates": 24, "population_size": 8,
}
VALIDATION_REPEATS = 5

# W&B controls match optuna_inference_optimization_lab_v2.
WANDB_ENABLED = True
WANDB_PROJECT = "inference-optimization"
WANDB_ENTITY = "maorblumberg-tel-aviv-university"  # or None for your SDK default
WANDB_MODE = "online"  # "offline" keeps runs local; False above disables the SDK
WANDB_GROUP = CAMPAIGN  # same label across framework VMs

# Fill with actual verified values. Never mark checks true without performing them.
PREFLIGHT = {
    framework: {
        "gpu_type": "REPLACE", "framework_version": "REPLACE",
        "cuda_version": "REPLACE", "driver_version": "REPLACE",
        "model_revision": "REPLACE_WITH_MODEL_COMMIT",
        "native_config_verified": True, "quality_verified": False,
        "verified_precision_combinations": [],  # weights/cache pairs approved on this VM
    }
    for framework in ["vllm", "sglang", "trtllm"]
}


## W&B login and study logging
Install the SDK with `pip install -e '.[notebook,wandb]'`. Run the login cell before creating studies, as in `optuna_inference_optimization_lab_v2.ipynb`. Use `WANDB_ENABLED=False` to disable tracking, or `WANDB_MODE="offline"` to save runs locally without login; upload offline runs later with `wandb sync`.

The existing shared study logger records sampled parameters (including precision), rendered engine configs, workload/environment contracts, benchmark/objective metrics, request accounting, failure/pruning details, cumulative GPU cost, and final best/Pareto candidates. One W&B run is created per study invocation, named after its stable Optuna study name and grouped by `WANDB_GROUP`. Synthetic runs carry the `SYNTHETIC` tag. W&B logging failures warn and allow optimization to continue. Independent validation remains in local artifacts, matching v2.

For unattended runs, authenticate on the VM first with `wandb login` or provide `WANDB_API_KEY` through the service environment. Keep credentials outside notebook cells and experiment configs. The batch supervisor authenticates before launching the queue; CPU smoke tests disable tracking.


In [ ]:
if WANDB_ENABLED and WANDB_MODE == "online":
    import wandb
    wandb.login()  # uses existing credentials/WANDB_API_KEY or prompts interactively


## Explicit base and extended search spaces
Every domain below is editable. Base numeric domains must be regularly spaced and contain at least two values. Extended categorical domains must be supported by the installed engine and model. Remove unsupported choices after preflight, including numerical quality checks for cache dtype changes. Graph profiles and padding are conditional. `schedule_conservativeness` is SGLang-only; Phase A retains its fixed baseline value of 1.0, while Phase B searches the explicit domain below.

In [ ]:
# Illustrative SLOs; replace after measuring the pinned model/workload/GPU.
TTFT_LIMITS_MS = {"1": 100.0, "128": 200.0}
E2E_LIMITS_MS = {"1": 2000.0, "128": 6000.0}
FAILURE_RATE_LIMIT = 0.0
CONSTRAINED = False
PRUNER_NAMES = ["nop", "median", "percentile", "halving", "hyperband", "threshold"]
PRUNING_SAMPLES = 32
# Ordinary serving is explicit. Setup is opt-in and may download checkpoints.
HF_SETUP = {
    "target_model_id": MODEL, "target_revision": "REPLACE_WITH_MODEL_COMMIT",
    "framework": "vllm", "enable_dflash": False,
    "speculative_algorithm": None, "draft_model_id": None, "draft_revision": None,
    "block_size": 16, "num_speculative_tokens": 15, "max_draft_len": 3,
}
# SGLang: DFLASH/DSPARK; vLLM: enable_dflash=True selects DSpark.
# TRT: explicit algorithms accepted by predefined.trtllm.validate_draft_request.
# Configure a separate campaign/run root to compare ordinary and speculative serving.
MODEL_PROFILES = {}  # populated only by explicit setup_pinned_models()
MODEL_SETUP_IDENTITY = None
MODEL_SETUP_CHOICES = None
PHASE_B_MODEL = None
PHASE_B_SNAPSHOT = None


In [ ]:
PREFILL_BUDGETS = list(range(1024, 8193, 1024))
TOKEN_BUDGETS = list(range(1024, 32769, 256))
CAPACITIES = list(range(8, 129, 8))
BASE_SPACES = {
    1: {
        "vllm": {"max_num_batched_tokens": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
        "sglang": {"chunked_prefill_size": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
        "trtllm": {"max_num_tokens": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
    },
    128: {  # shared regime for all concurrency levels > 1
        "vllm": {
            "max_num_seqs": CAPACITIES, "max_num_batched_tokens": TOKEN_BUDGETS,
            "gpu_memory_utilization": [0.75, 0.80, 0.85, 0.90],
        },
        "sglang": {
            "max_running_requests": CAPACITIES, "chunked_prefill_size": TOKEN_BUDGETS,
            "mem_fraction_static": [0.70, 0.75, 0.80, 0.85],
        },
        "trtllm": {
            "max_batch_size": CAPACITIES, "max_num_tokens": TOKEN_BUDGETS,
            "kv_cache_free_gpu_memory_fraction": [0.65, 0.75, 0.85, 0.95],
        },
    },
}

EXPANDED_CHOICES = {'vllm': {'async_scheduling': [False, True],
          'performance_mode': ['balanced', 'interactivity', 'throughput'],
          'attention_backend': ['auto', 'FLASH_ATTN', 'FLASHINFER'],
          'kv_cache_dtype': ['auto'],
          'enable_prefix_caching': [False, True],
          'graph_mode': ['NONE', 'PIECEWISE', 'FULL_AND_PIECEWISE']},
 'sglang': {'attention_backend': ['triton', 'fa3', 'flashinfer'],
            'disable_radix_cache': [False, True],
            'schedule_policy': ['fcfs', 'lpm'],
            'enable_mixed_chunk': [False, True],
            'disable_overlap_schedule': [False, True],
            'cuda_graph_backend_decode': ['disabled', 'full'],
            'cuda_graph_backend_prefill': ['disabled', 'breakable']},
 'trtllm': {'attn_backend': ['TRTLLM'],
            'enable_chunked_prefill': [False, True],
            'enable_block_reuse': [False, True],
            'capacity_scheduler_policy': ['GUARANTEED_NO_EVICT', 'MAX_UTILIZATION'],
            'graph_enabled': [False, True]}}

EXTRA_SPACES = {
    "vllm": {"graph_capture_profile": ["powers", "dense"]},
    "sglang": {
        "schedule_conservativeness": [0.5, 0.75, 1.0, 1.25, 1.5],
        "max_prefill_tokens": [8192, 16384, 24576, 32768],
    },
    "trtllm": {
        "graph_capture_profile": ["powers", "dense"],
        "graph_padding": [0, 1],
    },
}


## Optional parameters and measured-window telemetry
All optional dimensions below start disabled. Enable only a small remotely verified
subset before constructing a runner. They apply to **Phase B only**; A preserves
the original baseline. Single-value domains provide a fixed B override. For a
matching A/B context limit, also set `max_model_len` in both phases' fixed_server.
Changing options, telemetry or versioned implementation creates a new campaign hash.

| Optional parameter | Initial domain | Conditions / implications |
|---|---|---|
| long_prefill_token_threshold | 0, 128, 256 | C>1; per-request prefill cap; smaller chunks can worsen TTFT; hybrid alignment applies |
| prefill_schedule_interval | 1, 2 | C>1; decode-only intervening steps may delay new prompts |
| compilation_mode | 0, 3 | Distinct from CUDA graphs; mode 0 maps PIECEWISE to NONE and FULL_AND_PIECEWISE to FULL_DECODE_ONLY |
| flash_attn_max_num_splits_for_cuda_graph | 8, 16, 32 | Only explicit FLASH_ATTN with graphs active; actual kernel support must be verified |
| block_size | 16, 32 | Backend/hybrid-model constraints; inspect effective size remotely |
| max_model_len | 1024 | Must cover prompt+output; single-value fixed override in B |
| watermark | 0, .05, .10 | C>1; use only if cache-pressure/preemption logs justify reserving headroom |

Compilation comparisons should be separate campaigns with a constant compatible
graph policy. Fix precision/backend first; 20 attempted trials cannot cover all axes.
Existing prefix caching is not a controlled reuse experiment: warmup repeats the
first measured prompt. Use a separate repeated-prefix campaign to measure its benefit.

Telemetry matches SGLang v3 boundaries: measured requests only, excluding startup,
warmup and rest. Running/waiting counts come from server gauges, not client limits.
Counts sum independent engines; KV usage is the maximum engine fraction (0..1),
not GPU memory usage or necessarily attention-only state. GPU utilization is NVML
busy percent, not kernel occupancy. Set a verified physical NVML index; no device
is inferred from CUDA_VISIBLE_DEVICES. Unsupported metrics remain null with reasons.
Time averages use left holds over covered time, expiring after two sampling intervals.
Coverage and sampled peaks are recorded; publication cadence may limit resolution.
Collectors stop before server cleanup. JSONL, execution sidecars, Optuna attributes
and W&B retain metrics; failed/pruned attempts retain partial execution telemetry.

Source: [vLLM 0.33 scheduler](https://github.com/vllm-project/vllm/blob/v0.33.0/vllm/config/scheduler.py),
[compilation](https://github.com/vllm-project/vllm/blob/v0.33.0/vllm/config/compilation.py),
[attention](https://github.com/vllm-project/vllm/blob/v0.33.0/vllm/config/attention.py),
[Prometheus gauges](https://github.com/vllm-project/vllm/blob/v0.33.0/vllm/v1/metrics/loggers.py).


In [ ]:
# Enable only the additional dimensions verified on the remote GPU.
optional_params = copy.deepcopy(DEFAULT_OPTIONAL_PARAMS)
# Example: optional_params["long_prefill_token_threshold"]["enabled"] = True
# Example: optional_params["prefill_schedule_interval"]["enabled"] = True
TELEMETRY = {
    "enabled": True,
    "interval_s": 0.5,
    "timeout_s": 0.25,
    "gpu_index": None,  # Set verified physical NVML device index to collect GPU utilization.
}
print("Optional parameters:", optional_params)


## Bind the shared runner
Persistent SQLite storage is local to each VM. Copy artifacts off the VM periodically. A VM-wide file lock prevents overlapping workers under the same root. The runner records abandoned RUNNING trials as FAIL after acquiring exclusive ownership; failed and pruned attempts count toward the configured trial budget. Notebook kernels do not have the batch script’s process-group watchdog. After an interrupted notebook, ensure its server processes have stopped before resuming.
Every group invocation reads the current controls and resumes via SQLite. Editing the request map or sampler selection produces a separate campaign configuration hash.


## Independent weight and attention KV precision
Phase B samples weight/cache precision independently; A stays BF16/BF16.
The v3 adapter uses BF16 compute, `quantization=fp8_per_tensor` for online FP8
weights and `kv_cache_dtype=fp8` for FP8 attention KV. It removes the obsolete
`calculate_kv_scales` flag. Unit KV scales are not calibration: verify quality
against BF16 remotely. Qwen3.5 also has recurrent state, which is not all FP8 KV.

Official [online quantization](https://docs.vllm.ai/en/v0.33.0/features/quantization/online/)
and [KV quantization](https://docs.vllm.ai/en/v0.33.0/features/quantization/quantized_kvcache/).


In [ ]:
# Independent Phase B axes. To force FP8 weights + FP8 cache, use ["fp8"] for both.
# Phase A always uses unquantized BF16 weights and the default cache (BF16).
PRECISION_CHOICES = {
    "vllm": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
    "sglang": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
    "trtllm": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
}
# TRT on-load weight quantization needs calibration. Adjust for your pinned version;
# prepare/cache its calibration dataset before launching the unattended campaign.
TRT_FP8_CALIBRATION = {
    "calib_batches": 32, "calib_batch_size": 1,
    "calib_max_seq_length": 512, "tokenizer_max_seq_length": 512,
}


## JSON export and response capture
Run the next cell before constructing runners. It fixes JSON exports containing `inf`,
`-inf` or `nan` without modifying the stored Optuna trials. To finish the previous studies,
set `SAVE_REQUEST_OUTPUTS=False` and use exactly the previous campaign/model configuration.

With `SAVE_REQUEST_OUTPUTS=True` (the default here), generated text is retained after each
saved benchmark in `*-measured.jsonl`, under `results.generated_texts`. This setting changes
the frozen configuration and creates a fresh results directory under the same campaign.
Changing `RUN_NAME` alone does not reset studies. Text from previous runs cannot be recovered.
Responses are saved locally, not automatically uploaded to W&B, and are available after the
benchmark returns rather than streamed live. Attempts interrupted by errors or pruning may
not produce a result row. The random-token workload is for performance, not quality validation.


In [ ]:
# Notebook-only JSON export fix: preserves SQLite values; sanitizes exported copies.
import json
import math
from pathlib import Path
import llm_optimizer.experiment.phase1 as phase1_module
import llm_optimizer.experiment.workflow as workflow_module

def clean_export_json(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {key: clean_export_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_export_json(item) for item in value]
    return value

def safe_write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(clean_export_json(value), indent=2, allow_nan=False) + "\n"
    )
    temporary.replace(path)

# phase1 imported write_json by name, so patch both references in this kernel.
phase1_module.write_json = safe_write_json
workflow_module.write_json = safe_write_json


In [ ]:
# Compatibility is implemented in the versioned policy, not kernel monkeypatches.
VLLM033_COMPATIBILITY = {
    "adapter": "vllm-v3", "framework_version": "0.33.0",
    "online_weight_scheme": "fp8_per_tensor",
    "fp8_kv_scaling": "unit-scales-unless-checkpoint-calibrated",
}


In [ ]:
def make_runner(for_model_setup=False):
    # The versioned runner binds policy, launch and telemetry together.
    if "vllm" in FRAMEWORKS:
        PREFLIGHT["vllm"]["notebook_compatibility"] = dict(VLLM033_COMPATIBILITY)
    if MODEL_SETUP_IDENTITY is not None and MODEL_SETUP_IDENTITY != HF_SETUP:
        raise ValueError("HF_SETUP changed; clear MODEL_PROFILES and MODEL_SETUP_IDENTITY, then rerun setup")
    if MODEL_SETUP_CHOICES is not None and MODEL_SETUP_CHOICES != EXPANDED_CHOICES[HF_SETUP["framework"]]:
        raise ValueError("Expanded choices changed after HF setup; clear setup profiles and rerun setup")
    settings = {
        "ttft_limits_ms": TTFT_LIMITS_MS, "e2e_limits_ms": E2E_LIMITS_MS,
        "failure_rate_limit": FAILURE_RATE_LIMIT, "pruning_samples": PRUNING_SAMPLES,
        "constrained": CONSTRAINED,
        "wandb_enabled": WANDB_ENABLED,
        "wandb_project": WANDB_PROJECT, "wandb_entity": WANDB_ENTITY,
        "wandb_mode": WANDB_MODE, "wandb_group": WANDB_GROUP,
        "frameworks": FRAMEWORKS, "real_frameworks": FRAMEWORKS,
        "run_real_gpu": RUN_REAL_GPU and not for_model_setup, "quick_demo": False,
        "phases": ["A", "B"], "concurrencies": CONCURRENCIES,
        "seeds": SEEDS, "sampler_names": SAMPLER_NAMES, "pareto_samplers": PARETO_SAMPLERS,
        "phase_models": {"A": MODEL, "B": PHASE_B_MODEL or MODEL},
        "model_snapshots": {"A": MODEL_SNAPSHOT, "B": PHASE_B_SNAPSHOT or MODEL_SNAPSHOT},
        # Lab's phase-level fallback; actual study counts use the map below.
        "phase_requests": {"A": REQUESTS_BY_CONCURRENCY[128],
                           "B": REQUESTS_BY_CONCURRENCY[128]},
        "sampler_settings": SAMPLER_SETTINGS,
        "search_trials": {p: {"1": TRIALS, "128": TRIALS} for p in ["A", "B"]},
        "base_workload": {
            **({"output_details": True} if SAVE_REQUEST_OUTPUTS else {}),
            "dataset_name": "random", "apply_chat_template": True, "random_input_len": 512,
            "random_output_len": 128, "random_range_ratio": 1.0,
            "seed": 2026, "request_rate": "inf",
        },
        "validation_repeats": VALIDATION_REPEATS,
        "validation_requests": {"A": 1000, "B": 1000},
        "gpu_count": 1, "rest_seconds": 10,
        "real_preflight": {
            phase: {f: {**PREFLIGHT[f], "expanded_choices": EXPANDED_CHOICES[f],
                           **(MODEL_PROFILES.get(f, {}) if phase == "B" else {})}
                    for f in FRAMEWORKS}
            for phase in ["A", "B"]
        },
    }
    runner = VLLMRunnerV3(
        settings, BASE_SPACES, EXPANDED_CHOICES, EXTRA_SPACES, RUN_ROOT,
        precision_choices=PRECISION_CHOICES, trt_fp8_calibration=TRT_FP8_CALIBRATION,
        optional_params=optional_params, telemetry_options=TELEMETRY,
        requests_by_concurrency=REQUESTS_BY_CONCURRENCY,
        pruning_min_completed_requests=PRUNING_MIN_COMPLETED_REQUESTS,
    )

    original_logger = runner.lab.NotebookWandbLogger
    run_prefix = RUN_NAME

    def named_logger(study, contract):
        logger = original_logger(study, contract)
        if logger.tracker is not None and run_prefix:
            logger.tracker.run.name = f"{run_prefix}-{study.study_name}"
        return logger

    runner.lab.NotebookWandbLogger = named_logger
    return runner

# Construct from the current controls for each invocation; SQLite resumes studies.
def get_runner():
    return make_runner()

def run_objective_group(phase, target, concurrencies, pruner_name="nop", sampler_names=None, constrained=None):
    global LAST_RUN_DIR
    runner = get_runner()
    LAST_RUN_DIR = runner.run_dir
    print("Campaign:", CAMPAIGN, "| W&B run prefix:", RUN_NAME)
    print("Results directory:", LAST_RUN_DIR)
    print("Concurrency / full-trial requests:",
          [(c, REQUESTS_BY_CONCURRENCY[c]) for c in concurrencies])
    return runner.run_objective_group(
        phase, target, concurrencies, pruner_name, sampler_names=sampler_names,
        constrained=CONSTRAINED if constrained is None else constrained,
    )

def setup_pinned_models():
    """Explicit GPU setup; commit resolved profiles into the editable controls."""
    global PHASE_B_MODEL, PHASE_B_SNAPSHOT, MODEL_SETUP_IDENTITY, MODEL_SETUP_CHOICES
    if not RUN_REAL_GPU:
        raise ValueError("Set RUN_REAL_GPU=True before model setup")
    runner = make_runner(for_model_setup=True)
    runner.lab.RUN_REAL_GPU = True
    with runner.lock():
        summary = runner.lab.configure_hf_models(**HF_SETUP)
    framework = HF_SETUP["framework"]
    import copy
    profile = runner.lab.REAL_PREFLIGHT["B", framework]
    # Verification declarations remain editable in PREFLIGHT after setup.
    MODEL_PROFILES[framework] = {
        key: copy.deepcopy(profile[key]) for key in (
            "model_revision", "hf_target_model_id", "fixed_server", "draft_model_id",
            "draft_revision", "speculative_decoding", "framework_version", "cuda_version",
            "driver_version", "gpu_type",
        ) if key in profile
    }
    if profile["expanded_choices"] != EXPANDED_CHOICES[framework]:
        MODEL_PROFILES[framework]["expanded_choices"] = copy.deepcopy(profile["expanded_choices"])
    MODEL_SETUP_CHOICES = copy.deepcopy(EXPANDED_CHOICES[framework])
    PHASE_B_MODEL, PHASE_B_SNAPSHOT = summary["target_model_id"], summary["target_path"]
    MODEL_SETUP_IDENTITY = dict(HF_SETUP)
    return summary


## Resolve a pinned Hugging Face model for vLLM
Run these cells after the configuration and runner cells. Setup downloads or reuses the model snapshot and requires a CUDA GPU. Choose an immutable commit for reproducible reruns; `main` resolves to its current commit. The setup below selects ordinary serving.


In [ ]:
import tempfile
from pathlib import Path

hf_cache = Path(tempfile.mkdtemp(prefix="hf-cache-", dir="/tmp"))
HF_SETUP["cache_dir"] = str(hf_cache)

print("Hugging Face cache:", hf_cache)
print("Writable:", hf_cache.is_dir())

In [ ]:
verify_runtime()  # Check the exact release before model downloads.
FRAMEWORKS = ["vllm"]
RUN_REAL_GPU = True
MODEL_PROFILES = {}
MODEL_SETUP_IDENTITY = None
MODEL_SETUP_CHOICES = None

HF_SETUP.update(
    framework="vllm", target_model_id=MODEL, target_revision="main",
    enable_dflash=False, speculative_algorithm=None,
    draft_model_id=None, draft_revision=None,
)
hf_models = setup_pinned_models()
hf_models


In [ ]:
MODEL = hf_models["target_model_id"]
MODEL_SNAPSHOT = hf_models["target_path"]
for key in ("gpu_type", "framework_version", "cuda_version",
            "driver_version", "model_revision"):
    PREFLIGHT["vllm"][key] = MODEL_PROFILES["vllm"][key]
print("Pinned snapshot:", MODEL_SNAPSHOT)
print("Pinned revision:", PREFLIGHT["vllm"]["model_revision"])


In [ ]:
MODEL = hf_models["target_model_id"]
MODEL_SNAPSHOT = hf_models["target_path"]

# Record the resolved runtime and revision for Phase A.
for key in (
    "gpu_type",
    "framework_version",
    "cuda_version",
    "driver_version",
    "model_revision",
):
    PREFLIGHT["vllm"][key] = MODEL_PROFILES["vllm"][key]

print("Phase A snapshot:", MODEL_SNAPSHOT)
print("Phase B snapshot:", PHASE_B_SNAPSHOT)

## Remote runtime and native-command audit
This checks flag names for every selected value at each concurrency, including
optional conditional branches. It cannot validate nested schemas, backend/model
compatibility, effective graph fallbacks or output quality; smoke-test those remotely.


In [ ]:
import optuna
print(verify_runtime())
VLLM_HELP = runtime_help()
audit_runner = make_runner(for_model_setup=True)
audit_lab = audit_runner.lab

def notebook_vllm_command(lab, config, port=8000, concurrency=8):
    command = native_command(lab, config, MODEL_SNAPSHOT, concurrency, port)
    audit_command(command, VLLM_HELP)
    return command

for concurrency in CONCURRENCIES:
    menu = audit_lab.sampling_menu("vllm", concurrency, "expanded")
    print("Concurrency:", concurrency, "sampling menu:", menu)
    baseline = {key: values[0] for key, values in menu.items()}
    if "FLASH_ATTN" in menu["attention_backend"]:
        baseline["attention_backend"] = "FLASH_ATTN"
    baseline["graph_mode"] = next((v for v in menu["graph_mode"] if v != "NONE"), "NONE")
    if "compilation_mode" in menu and 3 in menu["compilation_mode"]:
        baseline["compilation_mode"] = 3
    for name, values in menu.items():
        for value in values:
            config = audit_lab.suggest_config(optuna.trial.FixedTrial({**baseline, name: value}),
                                              "vllm", concurrency, "expanded")
            config = {**MODEL_PROFILES["vllm"].get("fixed_server", {}), **config}
            notebook_vllm_command(audit_lab, config, concurrency=concurrency)
# Also audit each precision pair, even when optional domains are disabled.
for weights in PRECISION_CHOICES["vllm"]["weight_precision"]:
    for cache in PRECISION_CHOICES["vllm"]["kv_cache_precision"]:
        print(weights, cache, notebook_vllm_command(audit_lab, audit_lab.precision_config("vllm", weights, cache)))
print("Flag-name audit passed; GPU/backend/quality validation remains separate.")


In [ ]:
# Optional: starts ONE real server, requests text, then stops it; no Optuna trial.
RUN_VLLM033_SMOKE = False
SMOKE_WEIGHTS, SMOKE_CACHE = "fp8", "fp8"  # repeat with bf16/bf16 as a reference
SMOKE_PORT = 8000  # must be unused
if RUN_VLLM033_SMOKE:
    import requests
    from llm_optimizer.server_utils import start_server, terminate_process_top_down

    smoke_runner = make_runner(for_model_setup=True)
    smoke_lab = smoke_runner.lab
    smoke_config = smoke_lab.render_engine_config("vllm", 8, {
        "max_num_seqs": 8, "max_num_batched_tokens": 2048,
        "gpu_memory_utilization": 0.75,
    })
    smoke_config.update(smoke_lab.precision_config("vllm", SMOKE_WEIGHTS, SMOKE_CACHE))
    # Start with a small eager configuration and automatic attention selection.
    # Test the expanded backends/graphs/scheduling separately before enabling those domains.
    smoke_config.update(attention_backend="auto", enforce_eager=True,
                        async_scheduling=False, enable_prefix_caching=False)
    smoke_config.pop("compilation_config", None)
    smoke_config = {**MODEL_PROFILES["vllm"].get("fixed_server", {}), **smoke_config}
    smoke_command = notebook_vllm_command(smoke_lab, smoke_config, SMOKE_PORT)
    print(smoke_command)
    with smoke_runner.lock():
        process = None
        try:
            process = start_server(smoke_command, {},
                                   f"http://127.0.0.1:{SMOKE_PORT}/health", False)
            response = requests.post(f"http://127.0.0.1:{SMOKE_PORT}/v1/chat/completions",
                json={"model": str(MODEL_SNAPSHOT), "messages": [
                    {"role": "user", "content": "What is 17 times 19? Explain briefly."}],
                    "temperature": 0, "max_tokens": 256}, timeout=180)
            response.raise_for_status()
            print(response.json()["choices"][0]["message"])
            from llm_optimizer.experiment.telemetry_vllm_v3 import aggregate_prometheus
            metrics = requests.get(f"http://127.0.0.1:{SMOKE_PORT}/metrics", timeout=10)
            metrics.raise_for_status()
            print("Available gauges:", aggregate_prometheus(metrics.text))
        finally:
            if process is not None:
                terminate_process_top_down(process)
    print("Smoke completed. Review text; this does not approve the expanded search space.")


Set the native configuration and quality verification fields in `PREFLIGHT["vllm"]` only after checking them on this VM. List every verified weight/cache combination selected by `PRECISION_CHOICES`; narrow those choices if needed. Run the experiment cells after setup. Rerunning the initial controls resets the setup, so run these cells again. Notebook variables are not passed to the batch script.


In [ ]:
print("vLLM version:", MODEL_PROFILES["vllm"]["framework_version"])
print("Expanded choices:", EXPANDED_CHOICES["vllm"])

print("Enabled optional parameters:", {k: v for k, v in optional_params.items() if v["enabled"]})
# Select the precision combinations you intend to test.
PRECISION_CHOICES["vllm"] = {
    "weight_precision": ["fp8"],   # e.g. ["bf16", "fp8"]
    "kv_cache_precision": ["fp8"], # e.g. ["bf16", "fp8"]
}

# Change these to True only after checking native configuration support
# and testing that the model's generated output is acceptable.
PREFLIGHT["vllm"].update({
    "native_config_verified": False,
    "quality_verified": False,

    # List only combinations you verified on this VM.
    # Names use "weight_precision/kv_cache_precision".
    "verified_precision_combinations": [
        # "bf16/bf16",
        # "bf16/fp8",
        # "fp8/bf16",
        # "fp8/fp8",
    ],
})

print("Preflight:", PREFLIGHT["vllm"])
print("Precision choices:", PRECISION_CHOICES["vllm"])

## Independent experiment cells
Day 1: preflight, untuned baseline measurements, and recovery drills. Days 2–4: base (A) versus extended (B) searches on the same model, then median versus no pruning. Days 5–6: independent repeats and workload checks. Day 7: sustained-load confirmation. The queue below performs the search stage; use the validation cell afterwards. Measure trial duration during the pilot and adjust trial counts to fit the week. A finite completed queue exits successfully rather than repeating identical experiments indefinitely.

### A: single e2e (no pruning)

In [ ]:
A_single_e2e_studies = run_objective_group(
    "A", "e2e", [1], pruner_name="nop",
)
plot_group(A_single_e2e_studies)


### A: single ttft (no pruning)

In [ ]:
A_single_ttft_studies = run_objective_group(
    "A", "ttft", [1], pruner_name="nop",
)
plot_group(A_single_ttft_studies)


### A: loaded throughput (no pruning)

In [ ]:
A_loaded_throughput_studies = run_objective_group(
    "A", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(A_loaded_throughput_studies)


### A: loaded e2e (no pruning)

In [ ]:
A_loaded_e2e_studies = run_objective_group(
    "A", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(A_loaded_e2e_studies)


### B: single e2e (no pruning)

In [ ]:
B_single_e2e_studies = run_objective_group(
    "B", "e2e", [1], pruner_name="nop",
)
plot_group(B_single_e2e_studies)


### B: single ttft (no pruning)

In [ ]:
B_single_ttft_studies = run_objective_group(
    "B", "ttft", [1], pruner_name="nop",
)
plot_group(B_single_ttft_studies)


### B: loaded throughput (no pruning)

In [ ]:
B_loaded_throughput_studies = run_objective_group(
    "B", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(B_loaded_throughput_studies)


### B: loaded e2e (no pruning)

In [ ]:
B_loaded_e2e_studies = run_objective_group(
    "B", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(B_loaded_e2e_studies)


### B: loaded throughput with median pruning
The first ten trials establish pruning history. Decisions use measured request checkpoints, with at least three reference trials. The no-pruning comparison above uses the same workload and seeds.

In [ ]:
B_loaded_throughput_studies = run_objective_group(
    "B", "throughput", [c for c in CONCURRENCIES if c > 1],
    pruner_name="median",
)
plot_group(B_loaded_throughput_studies)


## View and export saved model responses
Run this cell after any experiment has saved a benchmark result. By default it shows every
response from the most recently modified result file's last benchmark. Select a file and row
to inspect an earlier trial. It also writes a readable text copy alongside the JSONL file.


In [ ]:
# Run after a benchmark finishes; rerun to inspect newer saved results.
# Optional: paste a prior campaign directory or an exact *-measured.jsonl path.
RESULTS_DIR = None
RESULT_FILE = None
RESULT_INDEX = -1  # last saved benchmark in the selected file
WRITE_READABLE_COPY = True

if RESULT_FILE is not None:
    output_path = Path(RESULT_FILE)
else:
    output_dir = Path(RESULTS_DIR) if RESULTS_DIR is not None else globals().get("LAST_RUN_DIR")
    if output_dir is None:
        raise ValueError("Run an experiment first, or set RESULTS_DIR / RESULT_FILE above")
    candidates = list(Path(output_dir).glob("*-measured.jsonl"))
    output_path = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

if output_path is None:
    print("No saved benchmark results yet.")
else:
    rows = []
    for line in output_path.read_text().splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print("Skipping an incomplete/invalid JSONL row; rerun after the writer finishes.")
    if not rows:
        print("No complete result rows yet.")
    else:
        row = rows[RESULT_INDEX]
        details = row.get("results", {})
        texts = details.get("generated_texts", [])
        print("File:", output_path, "| selected row:", RESULT_INDEX, "| rows:", len(rows))
        print("Trial:", row.get("optimization", {}).get("trial_number"))
        if not texts:
            print("This result contains no saved text; enable SAVE_REQUEST_OUTPUTS for future runs.")
        else:
            errors = details.get("errors", [])
            blocks = []
            for i, text in enumerate(texts):
                block = f"--- Request {i + 1} ---\n{text}"
                if i < len(errors) and errors[i]:
                    block += f"\nError: {errors[i]}"
                blocks.append(block)
            readable = "\n\n".join(blocks)
            print(readable)
            if WRITE_READABLE_COPY:
                row_number = RESULT_INDEX if RESULT_INDEX >= 0 else len(rows) + RESULT_INDEX
                readable_path = output_path.with_name(
                    f"{output_path.stem}-row{row_number}-generations.txt")
                readable_path.write_text(readable + "\n")
                print("Readable copy saved:", readable_path)


## Independent candidate validation
Run after search, within the remaining GPU budget. This uses the repository’s shortlist and independent validation workload/repeats; it is not an automatic sustained-load soak test. Keep the serving stack pinned and record any separate soak workload. Validation in this interactive cell does not use the batch watchdog.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    from llm_optimizer.experiment.workflow import load_studies, write_json
    runner = get_runner()
    with runner.lock():
        studies = load_studies(runner.lab)
        validation = runner.lab.validate_candidates(runner.lab.shortlist(studies))
        write_json(runner.run_dir / "validation.json", runner.lab.artifact_value(validation))


## Execution boundary
Use this notebook for v3 behavior. The batch Python entry point does not inherit
optional parameters, telemetry or the versioned precision adapter.
Notebook kernels do not have the batch process-group watchdog; after interruption,
verify that server processes have stopped before resuming.


## Optional Pareto, constraint, and pruning experiments
Run these cells independently after editing SLOs, samplers and pinned serving settings. Active pruning is supported only for unconstrained scalar objectives. Calibration uses the same task workload and scaled request checkpoints; pilot samples are additional to the 20-attempt comparison budgets. Diagnostics replay a first-checkpoint median rule and estimate savings; validate them on the GPU before trusting pruning. Threshold searches reuse a persisted task calibration.


In [ ]:
pareto_latency = run_objective_group("B", "pareto_latency", [1])
plot_group(pareto_latency)


In [ ]:
pareto_loaded = run_objective_group("B", "pareto_throughput", [c for c in CONCURRENCIES if c > 1])
plot_group(pareto_loaded)


In [ ]:
constrained_scalar = run_objective_group("B", "e2e", [1], constrained=True)
plot_group(constrained_scalar)


In [ ]:
constrained_pareto = run_objective_group("B", "pareto_latency", [1], constrained=True)
plot_group(constrained_pareto)


In [ ]:
diagnostics = get_runner().calibrate("B", "throughput", 8, FRAMEWORKS[0], samples=PRUNING_SAMPLES)
diagnostics


In [ ]:
pruning_comparison = {name: run_objective_group("B", "throughput", [8], pruner_name=name)
                      for name in PRUNER_NAMES}
